# Audit DB Intermediate: How to Join & Roll Up Across Tables

## Querying Across the Hierarchy with JOINs

**Difficulty: Intermediate | ~40 min | Requires Lab 2 (Modeling Runs, Spans, and Tool Calls) done in this Supabase project**

*Lab 3 of 8 in the Audit DB Labs module.*

Lab 2 ended by walking the run → span → tool_call / guardrail_event hierarchy with one query per level — five separate round trips to reconstruct a single run. That works, but it doesn’t scale: every span costs two more queries, and there’s no way to ask “which run had the most tool calls?” without pulling all the data into Python first.

This lab replaces the N+1 pattern with real SQL **JOINs** that reconstruct the same nested history in a single round trip, then layers on **aggregation** (“count tool calls per run”), **window functions** (“rank runs by span count”), and **EXPLAIN ANALYZE** to show the indexes at work.

Everything runs against your real Supabase Postgres, so every id and timestamp you see is genuine.

In [ ]:
!pip install python-dotenv==1.2.3 psycopg2-binary==2.9.12

One pinned line installs everything the lab needs, matching Section 9 exactly.

### Step 1 — Connect to Postgres

In [2]:
import os
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
import psycopg2

env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None)
load_dotenv(env_path)
database_url = os.getenv("DATABASE_URL")
assert database_url, "DATABASE_URL not found - complete README Section 7 setup first."

connection = psycopg2.connect(database_url, connect_timeout=10)
cursor = connection.cursor()
cursor.execute("SELECT version();")
print(cursor.fetchone()[0])

PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


Same connection pattern as Labs 1 and 2: walk upward for the module-level `.env`, load `DATABASE_URL`, connect with `psycopg2`, and print the version string as proof the connection works.

### Step 2 — Confirm Lab 2’s Schema Is Already Here

In [3]:
# Fail-fast: confirm all prerequisite tables exist
cursor.execute("SELECT to_regclass('public.run')")
assert cursor.fetchone()[0] is not None, "run table missing - complete Lab 1 first."
for table in ["span", "tool_call", "guardrail_event"]:
    cursor.execute("SELECT to_regclass(%s)", (f"public.{table}",))
    assert cursor.fetchone()[0] is not None, f"{table} table missing - complete Lab 2 first."
print("Lab 2 schema found: run, span, tool_call, guardrail_event.")

Lab 2 schema found: run, span, tool_call, guardrail_event.


Same fail-fast pattern Lab 2 used in its Step 2: `to_regclass` checks each table without raising an error if it’s missing. If any of these four tables isn’t here, stop and complete the prerequisite lab first — this lab’s JOINs assume the foreign keys and indexes Lab 2 built.

### Step 3 — Insert a Fresh Run to Query

In [4]:
# Insert a fresh run with the same shape as Lab 2 (2 spans, 1 tool_call, 2 guardrails)
cursor.execute(
    "INSERT INTO run (agent_name, status) VALUES (%s, %s) RETURNING run_id",
    ("support-agent", "running"),
)
lab3_run_id = cursor.fetchone()[0]

cursor.execute(
    "INSERT INTO span (run_id, span_name) VALUES (%s, %s) RETURNING span_id",
    (lab3_run_id, "retrieve_orders"),
)
span1_id = cursor.fetchone()[0]
cursor.execute(
    "INSERT INTO tool_call (span_id, tool_name, arguments, result, success) VALUES (%s, %s, %s, %s, %s)",
    (span1_id, "query_orders_db",
     '{"table": "orders", "filter": "shipped_yesterday"}', "1284 rows returned", True),
)
cursor.execute("UPDATE span SET ended_at = now() WHERE span_id = %s", (span1_id,))

cursor.execute(
    "INSERT INTO span (run_id, span_name) VALUES (%s, %s) RETURNING span_id",
    (lab3_run_id, "generate_answer"),
)
span2_id = cursor.fetchone()[0]
cursor.execute(
    "INSERT INTO guardrail_event (span_id, check_name, outcome, reason) VALUES (%s, %s, %s, %s)",
    (span2_id, "pii_scan", "pass", "no personal data found"),
)
cursor.execute(
    "INSERT INTO guardrail_event (span_id, check_name, outcome, reason) VALUES (%s, %s, %s, %s)",
    (span2_id, "prompt_injection_scan", "fail", "instruction-override attempt detected"),
)
cursor.execute("UPDATE span SET ended_at = now() WHERE span_id = %s", (span2_id,))

cursor.execute(
    "UPDATE run SET ended_at = now(), status = 'completed', total_cost = %s WHERE run_id = %s",
    (0.0073, lab3_run_id),
)
connection.commit()
print(f"Run {lab3_run_id} inserted with 2 spans, 1 tool_call, 2 guardrail_events.")

Run 39 inserted with 2 spans, 1 tool_call, 2 guardrail_events.


Same story as Lab 2: one `retrieve_orders` span with a tool call, one `generate_answer` span with two guardrail events (one pass, one fail). The data shape is identical to Lab 2’s — what changes is how we *query* it. We have a real run with real children in the database, so the JOIN queries below operate on genuine rows.

### Step 4 — From Five Queries to One: the LEFT JOIN

In [5]:
# Approach A: one query per level (the N+1 pattern Lab 2 used)
cursor.execute(
    "SELECT span_id, span_name FROM span WHERE run_id = %s ORDER BY started_at",
    (lab3_run_id,),
)
spans_a = cursor.fetchall()
count_a = 1
for span_id, span_name in spans_a:
    cursor.execute("SELECT count(*) FROM tool_call WHERE span_id = %s", (span_id,))
    count_a += 1
    cursor.execute("SELECT count(*) FROM guardrail_event WHERE span_id = %s", (span_id,))
    count_a += 1

# Approach B: one LEFT JOIN round trip
cursor.execute("""
    SELECT r.run_id, s.span_id, s.span_name,
           tc.tool_call_id, tc.tool_name, tc.result,
           ge.guardrail_event_id, ge.check_name, ge.outcome
    FROM run r
    LEFT JOIN span s ON s.run_id = r.run_id
    LEFT JOIN tool_call tc ON tc.span_id = s.span_id
    LEFT JOIN guardrail_event ge ON ge.span_id = s.span_id
    WHERE r.run_id = %s
    ORDER BY s.started_at, tc.tool_call_id, ge.guardrail_event_id
""", (lab3_run_id,))
rows_b = cursor.fetchall()

print(f"Approach A (N+1): {count_a} round trips to the database")
print(f"Approach B (JOIN):  1 round trip, {len(rows_b)} row(s) returned")
print()
print(f"Run {lab3_run_id} reconstructed via LEFT JOIN:")
for run_id, span_id, span_name, tc_id, tc_name, tc_result, ge_id, ge_name, ge_outcome in rows_b:
    print(f"  span {span_id} - {span_name} | tool_call {tc_id}: {tc_name} -> {tc_result} | guardrail {ge_id}: {ge_name} -> {ge_outcome}")

Approach A (N+1): 5 round trips to the database
Approach B (JOIN):  1 round trip, 3 row(s) returned

Run 39 reconstructed via LEFT JOIN:
  span 11 - retrieve_orders | tool_call 7: query_orders_db -> 1284 rows returned | guardrail None: None -> None
  span 12 - generate_answer | tool_call None: None -> None | guardrail 10: pii_scan -> pass
  span 12 - generate_answer | tool_call None: None -> None | guardrail 11: prompt_injection_scan -> fail


The `LEFT JOIN` preserves every span even when one side has no matching children: `retrieve_orders` has a tool call but no guardrail events (so `guardrail None: None -> None` appears), and `generate_answer` has guardrail events but no tool calls (so `tool_call None: None -> None`). A regular `INNER JOIN` would drop those spans entirely. One round trip replaces five, and the database does the matching internally instead of shipping rows back to Python one level at a time.

### Step 5 — Aggregation: Cost and Counts per Run

In [6]:
# Aggregate: count spans, tool_calls, and guardrail_events per run
cursor.execute("""
    SELECT r.run_id, r.agent_name, r.total_cost,
           count(DISTINCT s.span_id) AS spans,
           count(DISTINCT tc.tool_call_id) AS tool_calls,
           count(DISTINCT ge.guardrail_event_id) AS guardrail_events
    FROM run r
    LEFT JOIN span s ON s.run_id = r.run_id
    LEFT JOIN tool_call tc ON tc.span_id = s.span_id
    LEFT JOIN guardrail_event ge ON ge.span_id = s.span_id
    WHERE r.agent_name = 'support-agent' AND r.total_cost IS NOT NULL
    GROUP BY r.run_id, r.agent_name, r.total_cost
    ORDER BY r.run_id
""")
print("Run ID | Agent          | Cost    | Spans | Tools | Guardrails")
print("-" * 62)
for row in cursor.fetchall():
    print(f"{row[0]:>6} | {row[1]:<14} | ${row[2]:.4f} | {row[3]:>5} | {row[4]:>5} | {row[5]:>10}")

Run ID | Agent          | Cost    | Spans | Tools | Guardrails
--------------------------------------------------------------
     8 | support-agent  | $0.0042 |     0 |     0 |          0
    37 | support-agent  | $0.0057 |     2 |     1 |          2
    38 | support-agent  | $0.0057 |     2 |     1 |          2
    39 | support-agent  | $0.0073 |     2 |     1 |          2


`GROUP BY r.run_id` collapses every child row into one summary line per run. `count(DISTINCT ...)` avoids double-counting: a run with 2 spans and 2 guardrail events produces 4 joined rows (each guardrail paired with each span), but `DISTINCT` counts each child entity only once. Run 8 was the run Lab 1 created before Lab 2 built the child tables — it has zero spans, which is why its counts are all zero.

### Step 6 — Window Functions: Ranking Runs

In [7]:
# Window function: rank runs by span count (ties share the same rank)
cursor.execute("""
    SELECT run_id, agent_name, total_cost, spans,
           rank() OVER (ORDER BY spans DESC) AS span_rank
    FROM (
        SELECT r.run_id, r.agent_name, r.total_cost,
               count(DISTINCT s.span_id) AS spans
        FROM run r
        LEFT JOIN span s ON s.run_id = r.run_id
        WHERE r.agent_name = 'support-agent' AND r.total_cost IS NOT NULL
        GROUP BY r.run_id, r.agent_name, r.total_cost
    ) sub
    ORDER BY span_rank
""")
print("Rank | Run ID | Agent          | Cost    | Spans")
print("-" * 55)
for row in cursor.fetchall():
    print(f"  {row[4]:<3} | {row[0]:>6} | {row[1]:<14} | ${row[2]:.4f} | {row[3]:>5}")

Rank | Run ID | Agent          | Cost    | Spans
-------------------------------------------------------
  1   |     37 | support-agent  | $0.0057 |     2
  1   |     38 | support-agent  | $0.0057 |     2
  1   |     39 | support-agent  | $0.0073 |     2
  4   |      8 | support-agent  | $0.0042 |     0


A **window function** computes across ordered rows without collapsing them — unlike `GROUP BY`, it adds a column to each row rather than summarizing it away. `rank() OVER (ORDER BY spans DESC)` assigns rank 1 to the runs with the most spans; runs tied on span count share the same rank (all three at rank 1), and the next rank skips to 4 (not 2), which is what `rank()` does with ties. The inner subquery reuses the same `GROUP BY` pattern from Step 5; the window function wraps it to add the ranking column.

### Step 7 — EXPLAIN ANALYZE: Seeing the Indexes at Work

In [8]:
# EXPLAIN ANALYZE: see how Postgres uses the indexes from Lab 2
cursor.execute("""
    EXPLAIN ANALYZE
    SELECT r.run_id, s.span_name, tc.tool_name, ge.check_name, ge.outcome
    FROM run r
    LEFT JOIN span s ON s.run_id = r.run_id
    LEFT JOIN tool_call tc ON tc.span_id = s.span_id
    LEFT JOIN guardrail_event ge ON ge.span_id = s.span_id
    WHERE r.run_id = %s
    ORDER BY s.started_at
""", (lab3_run_id,))
for row in cursor.fetchall():
    print(row[0])

Sort  (cost=33.14..33.16 rows=5 width=140) (actual time=0.050..0.052 rows=3 loops=1)
  Sort Key: s.started_at
  Sort Method: quicksort  Memory: 25kB
  ->  Nested Loop Left Join  (cost=1.59..33.09 rows=5 width=140) (actual time=0.039..0.045 rows=3 loops=1)
        ->  Nested Loop Left Join  (cost=1.44..19.82 rows=5 width=80) (actual time=0.031..0.035 rows=2 loops=1)
              ->  Nested Loop Left Join  (cost=1.29..6.56 rows=5 width=48) (actual time=0.021..0.023 rows=2 loops=1)
                    ->  Seq Scan on run r  (cost=0.00..1.01 rows=1 width=4) (actual time=0.012..0.013 rows=1 loops=1)
                          Filter: (run_id = 39)
                          Rows Removed by Filter: 3
                    ->  Bitmap Heap Scan on span s  (cost=1.29..5.50 rows=5 width=48) (actual time=0.008..0.008 rows=2 loops=1)
                          Recheck Cond: (run_id = 39)
                          Heap Blocks: exact=1
                          ->  Bitmap Index Scan on idx_span_run_id  

Three things to notice in the plan: (1) `Bitmap Index Scan on idx_span_run_id` — Postgres uses the index Lab 2 created on `span.run_id` to jump straight to our two spans instead of scanning the whole table. (2) `Index Scan using idx_tool_call_span_id` and `idx_guardrail_event_span_id` — the child-table indexes do the same for each span’s tool calls and guardrail events. (3) The `Nested Loop Left Join` shape — Postgres nests the index lookups inside each other, starting from the `run` row and drilling into children via indexes. `actual time` under 0.1 ms confirms the indexes are doing their job; without them, the planner would fall back to sequential scans on every table.

### Step 8 — Recap

In [9]:
cursor.close()
connection.close()
print(f"Run {lab3_run_id}: JOIN queries + aggregation + window function - analysis complete.")

Run 39: JOIN queries + aggregation + window function - analysis complete.


Close the connection, then state what this lab proved: the same nested history Lab 2 reconstructed with five separate queries now comes back in one JOIN, and the same data is immediately available for aggregation, ranking, and performance analysis without pulling everything into Python first.

Reconstructing a full trace across `run` -> `span` -> `tool_call` is exactly how you answer the harness's core debugging question -- "what did my agent actually do on this run?" -- which is why these JOINs sit at the heart of any audit layer built for a real AI harness.
